In [ ]:
import pandas as pd

file = 'Survived.csv'
df = pd.read_csv(file)
print(f'Head contents of {file} is \n{df.head(10)}')
print(f'Label of {file} : {df.columns}')
print(f'Matrix size : {df.shape}')


In [ ]:
# ユニークな値を調べる
import matplotlib.pyplot as plt
%matplotlib inline

# for col in df.columns:
#    print(f'Unique item of {col} = {df[col].unique()}\n')
# i = 1; print(f'Unique item of {df.columns[i]} = {df[i].unique()}\n')
for i in {1,2,3,5,6,10}:
    print(f'Unique item of {df.columns[i]} = {df[df.columns[i]].unique()}')
    value_counts = df[df.columns[i]].value_counts()
    print(f'Each count of unique item of {df.columns[i]}:{value_counts}')

    value_counts.plot(kind='bar')
    plt.show()
    # plt.close()


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(5,3))
plt.xlabel('Value')
plt.ylabel('Frequency')

features = ['Age', 'Fare']

for feature in features:
    # plt.xlabel(f'{feature}')
    plt.title(f'Histgram of {feature}')
    df[feature].plot(kind='hist', bins=10, edgecolor='black')
    plt.show()
    plt.close()



In [ ]:
print('データのサイズの確認：',df.shape)
print('欠損値の数を確認：\n',df.isnull().sum())

In [ ]:
label = 'Age'
print(f'{label}の平均値=',df[label].mean())
print(f'{label}の中央値=',df[label].median())
print(f'{label}の標準偏差=',df[label].std())
print(f'{label}の最小=',df[label].min())
print(f'{label}の最大=',df[label].max())
print(f'{label}の最頻値=',df[label].mode())
df[label]


In [ ]:

# 'Age'の欠損値をその列の平均値で穴埋め
import numpy as np
df2 = pd.read_csv(file)
df2[label] = df[label].fillna(df[label].mean())
df2[label]

In [ ]:
label = 'Embarked'
print(f'{label}の最頻値=',df[label].mode())
# 'Embarked'の欠損値をその列の最頻値で穴埋め
df[label] = df[label].fillna(df[label].mode()[0])



In [ ]:
print('欠損値の数を確認：\n',df.isnull().sum())

In [ ]:
# 特徴量と正解データに分割する
col = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
x = df[col]
t = df['Survived']
print('x=\n',x.head(3),'\nt=\n',t.head(3))

In [ ]:
# ホールドアウト法により訓練データとテストデータに分割する
from sklearn.model_selection import train_test_split

# x_train, y_train：学習に利用する訓練データ
# x_test, y_test：検証に利用するテストデータ
# test_size　: 検証に利用する割合、注意：整数値を指定するとテストデータ件数とみなされる
x_train, x_test, y_train, y_test = train_test_split(x,t,test_size=0.2, random_state=0)
print("学習用訓練データの特徴量の行列サイズ：",x_train.shape)
print(x_train.head(3), "\n")
print("検証用テストデータの特徴量の行列サイズ：",x_test.shape)
print(x_test.head(3),"\n")
print("学習用訓練データの正解の行列サイズ：",y_train.shape)
print(y_train.head(3), "\n")
print("検証用テストデータの正解の行列サイズ：",y_test.shape)
print(y_test.head(3),"\n")

In [ ]:
# モデルの作成と学習
from sklearn import tree

model = tree.DecisionTreeClassifier(max_depth=5, 
            random_state=0, class_weight='balanced')
model.fit(x_train, y_train)
print('テストデータに対する正解率：', model.score(X=x_test, y=y_test ))

In [ ]:
# モデルチューニングの繰り返し作業のため学習用の関数を定義する

# def learn(x, t, depth=3):
def learn(x, t, depth=3, weight = 'balanced'):
    x_train, x_test, y_train, y_test = train_test_split(
        x,t,test_size=0.2, random_state=0)

    #  criterion : gini=ジニ不純度(既定), entropy=情報利得), log_loss=ログ損失(分類の確率的評価))
    #　splitter ： best=最良の分割を探索, random=ランダムに候補を選び、その中で最良を選択（高速化向け）
    model = tree.DecisionTreeClassifier(max_depth=depth, 
            # random_state=0, class_weight = 'balanced' )
            random_state=0, class_weight = weight )
    model.fit(x_train, y_train)

    score = model.score(X = x_train, y = y_train)
    score2 = model.score(X = x_test, y = y_test)
    return round(score,3), round(score2,3), model


In [ ]:
# 木の深さによる正解率の変化を確認
df_score = pd.DataFrame(columns = ['tree_depth', 'train_score', 'test_score'])

for j in range(1,15):
    train_score, test_score, model = learn(x, t, depth = j)
    df_score.loc[len(df_score)]=[j,train_score,test_score]
    sentence = '訓練データの正解率{}'
    sentence2 = 'テストデータの正解率{}'
    total_sentence = '深さ{}：'+sentence+', '+sentence2
    # print(total_sentence.format(j, train_score, test_score))

print('df_score = \n', df_score)




In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(8,5))
plt.plot(df_score['tree_depth'], df_score['train_score'], marker='o')
plt.plot(df_score['tree_depth'], df_score['test_score'], marker='s')
plt.xlabel('tree_depth')
plt.ylabel('score')
plt.title('Line Plot of Score vs tree_depth')
plt.legend(loc='upper left', frameon=True)
# bbox_to_anchor=(1,1)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
df3 = pd.read_csv('Survived.csv')
label = 'Age'
print(f'{label}の平均値=',df3[label].mean())
print(f'{label}の中央値=',df3[label].median())
print(f'{label}の標準偏差=',df3[label].std())
print(f'{label}の最小=',df3[label].min())
print(f'{label}の最大=',df3[label].max())
print(f'{label}の最頻値=',df3[label].mode())
df3[label]


In [ ]:
# 小グループ作成の基準となる列を指定
labels = ['Survived', 'Pclass']
for label in labels:
    print(f'Mean Age of Group {label}:',df3.groupby(label)['Age'].mean())


In [ ]:
# ピボットテーブルによる集計(最大値)
table0 = pd.pivot_table(df3, index = 'Survived', columns = 'Pclass', values = 'Age', aggfunc='max'); print('Pivot Table\n',table0)
print()
# ピボットテーブルによる集計(平均値)
table = pd.pivot_table(df3, index = 'Survived', columns = 'Pclass', values = 'Age'); print('Pivot Table\n',table)


In [ ]:
# ピボットテーブルの配列を調べる
import numpy as np
unqs1 = df3['Survived'].unique(); print('Uniques : ', unqs1)
sorted_unqs1  = np.sort(unqs1); print('Sorted Uniques : ', sorted_unqs1)
unqs2 = df3['Pclass'].unique(); print('Uniques : ', unqs2)
sorted_unqs2  = np.sort(unqs2); print('Sorted Uniques : ', sorted_unqs2)

print('\nValues of Pivot Table')
for j in range(table.shape[1]):
    print(f'{sorted_unqs2[j]} ', end='')
print()
for i in range(table.shape[0]):
    print(f'{sorted_unqs1[i]} : ',end='')
    for j in range(table.shape[1]):
        print(f'{table.values[i,j]:.3f} ',end='')
    print()


In [ ]:
#Age列の欠損値を確認
is_null = df3['Age'].isnull(); print('is_null=',is_null,'\n')

#ピボットテーブルで調べた、SurvivedとPclassの値の違いによる平均年齢の違いを用いてAge列の欠損値を穴埋めする
items = ['Survived', 'Pclass']
for i in range(len(sorted_unqs1)):
    for j in range(len(sorted_unqs2)):
        print(f'{i}:{items[0]}={sorted_unqs1[i]},{j}:{items[1]}={sorted_unqs2[j]} :: Age = {table.values[i,j]}')
        df3.loc[(df3[items[0]] == sorted_unqs1[i] ) & (df3[items[1]] == sorted_unqs2[j]) & (is_null), 'Age'] = table.values[i,j]

In [ ]:
col = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
x = df3[col]
t = df3['Survived']

# 木の深さによる正解率の変化を確認
df_score2 = pd.DataFrame(columns = ['tree_depth', 'train_score', 'test_score'])

for j in range(1,15):
    s1, s2, m = learn(x, t, depth = j)
    df_score2.loc[len(df_score2)]=[j,s1,s2]
    sentence = '訓練データの正解率{}'
    sentence2 = 'テストデータの正解率{}'
    total_sentence = '深さ{}：'+sentence+', '+sentence2
    # print(total_sentence.format(j, s1, s2))

print('df_score = \n', df_score2)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(8,5))
plt.plot(df_score2['tree_depth'], df_score2['train_score'], marker='o')
plt.plot(df_score2['tree_depth'], df_score2['test_score'], marker='s')
plt.xlabel('tree_depth')
plt.ylabel('score')
plt.title('Line Plot of Score vs tree_depth')
plt.legend(loc='upper left', frameon=True)
# bbox_to_anchor=(1,1)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
sex = df3.groupby('Sex')['Survived'].mean(); print(f'男女の比率：{sex}')
sex.plot(kind='bar')

In [ ]:
# 'Sex'を特徴量として追加してモデルの再学習を行う
col = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex']
x = df3[col]
t = df3['Survived']

train_score, test_score, model = learn(x, t)


In [ ]:
# Sexに対応した数値の特徴量'Sex_Value'を作る
df3.loc[(df3['Sex'] == 'female' ) , 'Sex_Value'] = 0
df3.loc[(df3['Sex'] == 'male' ) , 'Sex_Value'] = 1
df3.head(5)

In [ ]:
# get_dummies関数を用いて文字列を数値に変換する
items = ['Sex', 'Embarked']

for item in items:
    valued_item = pd.get_dummies(df3[item], dtype = int); print(valued_item.head(5))
    valued_item = pd.get_dummies(df3[item], drop_first = True, dtype = int); print(valued_item.head(5))


In [ ]:
# get_dummies関数を用いて文字列を数値に変換して元のデータフレームに横方向に連結する
male = pd.get_dummies(df3['Sex'], drop_first = True, dtype = int); print(male.head(5))

x_temp = pd.concat([x,male], axis = 1); x_temp.head(5)

In [ ]:
x_new = x_temp.drop('Sex', axis = 1); print(x_new.head(5))

# 木の深さによる正解率の変化を確認
df_score2 = pd.DataFrame(columns = ['tree_depth', 'train_score', 'test_score'])

for j in range(1,15):
    s1, s2, m = learn(x_new, t, depth = j)
    df_score2.loc[len(df_score2)]=[j,s1,s2]
    sentence = '訓練データの正解率{}'
    sentence2 = 'テストデータの正解率{}'
    total_sentence = '深さ{}：'+sentence+', '+sentence2
    # print(total_sentence.format(j, s1, s2))

print('df_score = \n', df_score2)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(8,5))
plt.plot(df_score2['tree_depth'], df_score2['train_score'], marker='o')
plt.plot(df_score2['tree_depth'], df_score2['test_score'], marker='s')
plt.xlabel('tree_depth')
plt.ylabel('score')
plt.title('Line Plot of Score vs tree_depth')
plt.legend(loc='upper left', frameon=True)
# bbox_to_anchor=(1,1)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 木の深さを8にして再学習してモデルを保存
s1, s2, model = learn(x_new, t, depth = 8)

df_feat = pd.DataFrame(model.feature_importances_, index = x_new.columns)
print('決定木分析における特徴量重要度を確認', df_feat)
df_feat.plot(kind='bar')
plt.show()

# 学習モデルを保存
import pickle
with open('Survived.pkl','wb') as f:
    pickle.dump(model, f)

In [ ]:
print('分岐条件の列を表示する:\n',model.tree_.feature)
print('分岐条件の閾値を含む配列を表示する:\n',model.tree_.threshold)
# リーフ(葉)に到達したデータの数を返す
print("各ノードに到達した数",model.tree_.weighted_n_node_samples)
print("モデル出力(Survived)の種類",model.classes_)

for i, feature in enumerate(model.tree_.feature):
    if feature == -2:
        print("ノード番号",i,"に到達した割合",model.tree_.value[i]*model.tree_.weighted_n_node_samples[i])

In [ ]:
# plot_tree関数で決定木を描画する

from sklearn.tree import plot_tree
plot_tree(model, feature_names=x_new.columns,filled=True)

In [ ]:
# 不均衡データに対するオプションClass_Weightの違いを確認
df_score2 = pd.DataFrame(columns = ['tree_depth', 'train_score', 'test_score'])

weights = [None, 'balanced']
for weight in weights:
    for j in range(1,15):
        # s1, s2, m = learn(x_new, t, j)
        s1, s2, m = learn(x_new, t, depth=j, weight=weight)
        df_score2.loc[len(df_score2)]=[j,s1,s2]
        sentence = '訓練データの正解率{}'
        sentence2 = 'テストデータの正解率{}'
        total_sentence = '深さ{}：'+sentence+', '+sentence2
        # print(total_sentence.format(j, s1, s2))

    print('df_score = \n', df_score2)